In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.preprocessing import StandardScaler,MinMaxScaler,FunctionTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [3]:
TRAIN_PATH = Path.cwd() / "train"
TEST_PATH = Path.cwd() / "test"

In [4]:
# Load Training set 
X_train = pd.read_csv(TRAIN_PATH / "X_train.csv")
y_train = pd.read_csv(TRAIN_PATH / "y_train.csv")

# Load Testing set
X_test = pd.read_csv(TEST_PATH / "X_test.csv")
y_test = pd.read_csv(TEST_PATH / "y_test.csv")


# Will later be updated by a try-catch block

In [ ]:
"""
Pass-through / Drop: cb_person_cred_hist_length (drop)
Log/Sqrt Transformed + Scaled: person_income (Log), loan_amnt (Sqrt), loan_percent_income (Log)
Custom Transformed: person_emp_exp (Zero-inflated custom transform)
Binned / Discretized: loan_int_rate
MinMax Scaled (As-is): credit_score, person_age (or Yeo-Johnson scaled)
"""

# Pipelining Step

In [ ]:
# We will make a column transformer for the encoders and create a custom Mapper function for our custom mapping cols
# Also I have decided to include the scalling in this transformer itself for a smoother flow.
# We will apply the scaler on the numeric columns before encoding in column transformer

In [ ]:
X_train.select_dtypes(include="number").head()

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score
35300,31.0,44485.0,4,8000.0,8.83,0.18,8.0,668
9746,24.0,35851.0,0,8725.0,12.18,0.24,2.0,645
26766,28.0,121251.0,6,25000.0,11.01,0.21,6.0,569
12944,23.0,76537.0,1,12000.0,14.96,0.16,4.0,538
466,24.0,53633.0,4,23975.0,11.01,0.45,2.0,678


In [ ]:
# Also for the cols : 
""" 
    Which have a meaningful boundary we cannot standardscaler as it makes value mean centered but can be negative
    For such cols we use MinMaxScaller instead that preserves the boundary of a column
"""

In [ ]:
# Now i split these into two lists to make sure our pipeline remains clean

In [ ]:
min_max_cols = ["person_age","person_emp_exp","cb_person_cred_hist_length","credit_score"]
std_cols = ['person_income', 'loan_amnt', 'loan_int_rate', 'loan_percent_income']

In [ ]:
# Now handling object columns

In [ ]:
X_train.select_dtypes(include="object").head()

,person_gender,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
35300,female,Associate,RENT,PERSONAL,Yes
9746,female,Master,RENT,MEDICAL,Yes
26766,female,High School,MORTGAGE,VENTURE,No
12944,female,High School,RENT,MEDICAL,No
466,male,Associate,RENT,HOMEIMPROVEMENT,No


In [ ]:
# To make sure the order for mapping cardinality values using OrdinalEncoder we will pass an explicit order to control behaviour
gender_order = ['female', 'male'] # Will map first value as 0 then next as 1 and so on...
file_order = ['No', 'Yes']

# Also the columns that we use OrdinalEncoder upon are : 
ord_cols = ["person_gender","previous_loan_defaults_on_file"]

In [ ]:
# Now OneHotEncoded cols will be as decided : 
onc_cols = ["person_home_ownership","loan_intent"]

In [ ]:
# Now handling our heirarchical col : person_education
heir_cols = ['person_education']

In [ ]:
# Now creating a custom mapper for person_education
def home_ownership_mapper(X):

    hierarchy_map = {
                            'High School': 1,
                            'Associate': 2,
                            'Bachelor': 3,
                            'Master': 4,
                            'Doctorate' : 5
    }

    # 1. Convert to a NumPy array so the format is always consistent
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy()
    
    # 2. Create a vectorized version of your dictionary lookup.
    # The .get(val, 0) handles missing values or typos by defaulting to 0.
    vector_lookup = np.vectorize(lambda val: hierarchy_map.get(val, 0))
    
    # 3. Apply it. This outputs the exact same 2D shape that came in.
    return vector_lookup(X)

# Wrap it up safely
home_ownership_transformer = FunctionTransformer(home_ownership_mapper, validate=False)

In [ ]:
# Column Transformer Sequential arranged 
col_transformer = ColumnTransformer(
        transformers = [
                            ("numeric_scale_min_max",MinMaxScaler(),min_max_cols),
                            ("numeric_scale_std",StandardScaler(),std_cols),
                            ("obj_enc_ord",OrdinalEncoder(
                                                                categories=[gender_order,file_order],
                                                                handle_unknown='use_encoded_value',
                                                                unknown_value=-1
                            ),ord_cols),
                            ("obj_enc_onc",OneHotEncoder(drop='first',handle_unknown='error'),onc_cols),
                            ("obj_enc_heir",home_ownership_transformer,heir_cols)
        ],
        remainder='drop'
)

In [ ]:
# Create the final pipeline
lor_pipeline = Pipeline(steps=[
    ('preprocessor', col_transformer),
    ('lor', LogisticRegression(max_iter=1000))
])

# Fit everything 
lor_pipeline.fit(X_train, y_train)